In [15]:
import pandas as pd

In [16]:
import numpy as np

In [17]:
import itertools

In [18]:
from collections import Counter

In [19]:
deployments = pd.read_csv("../data/raw/deployments.csv") 

In [20]:
proposals = pd.read_csv(
    "../data/raw/proposals.csv",
    low_memory=False # Disable low memory mode to avoid DtypeWarning  
) # If this doesn't work, almost certainlye the CSV files aren't in data/raw/

In [21]:
deployments["platform"].value_counts()

platform
snapshot    71952
tally        4394
realms       4224
daohaus      3459
aragon       2402
daostack       58
Name: count, dtype: int64

In [22]:
proposals["platform"].value_counts()

platform
snapshot    244886
daohaus      46936
tally        37839
realms       21153
aragon       15746
daostack      3573
Name: count, dtype: int64

In [23]:
deployments["deployment_id"].duplicated().sum() # Is there any duplicate deployment_id?

np.int64(0)

In [24]:
pd.crosstab(
    deployments["platform"],
    deployments["estimated_vp"].isna()
) #For each platform, how many deployments have missing 'estimated_vp' values?

estimated_vp,False,True
platform,,
aragon,2402,0
daohaus,3459,0
daostack,58,0
realms,4224,0
snapshot,0,71952
tally,4394,0


In [25]:
np.isinf(deployments["estimated_vp"]).sum() # From the reamianing deployments, how many have infinite 'estimated_vp' values? 

np.int64(13)

In [26]:
(deployments["estimated_vp"] == 0).sum() # From the remaining deployments, how many have a value of 0 for 'estimated_vp'?

np.int64(9254)

In [27]:
pd.crosstab(
    proposals["deployment_id"].isna(),
    proposals["platform_deployment_id"].isna()
) # Is 'platform_deployment_id' missing when 'deployment_id' is missing?

platform_deployment_id,False,True
deployment_id,,
False,0,350021
True,20112,0


In [ ]:
missing_id = proposals[proposals["deployment_id"].isna()]

In [29]:
missing_id.shape

(20112, 8)

In [ ]:
missing_id["platform"].value_counts() # Which platforms are missing 'deployment_id' in the proposals dataset?

platform
snapshot    20112
Name: count, dtype: int64

In [31]:
missing_id["platform_deployment_id"].notna().sum()

np.int64(20112)

In [ ]:
missing_id["platform_deployment_id"].isin(deployments["deployment_id"]).sum() # Are the 'platform_deployment_id' values in the proposals dataset that are not missing also present in the deployments dataset?

np.int64(20112)

In [ ]:
missing_id["platform_deployment_id"].head() # Show the first few 'platform_deployment_id' values in the missing 'deployment_id' dataset

332574    cakevote.eth
332575    cakevote.eth
332576    cakevote.eth
332577    cakevote.eth
332578    cakevote.eth
Name: platform_deployment_id, dtype: str

In [ ]:
deployments[deployments["platform"] == "snapshot"]["deployment_id"].head() # Show the first few 'deployment_id' values in the deployments dataset for the "snapshot" platform

10143    arbitrumfoundation.eth
10144               aavedao.eth
10145          snapshot.dcl.eth
10146              balancer.eth
10147                   cvx.eth
Name: deployment_id, dtype: str

In [ ]:

proposals[proposals["total_vp"].isna()]["platform"].value_counts() # Which platforms have missing 'total_vp' values in the proposals dataset?   

platform
snapshot    20112
Name: count, dtype: int64

In [63]:
# Create a new column 'dao_id' in the proposals dataset, which will be filled with 'deployment_id' values where available, and 'platform_deployment_id' values where 'deployment_id' is missing.
proposals["dao_id"] = proposals["deployment_id"].fillna(
    proposals["platform_deployment_id"]
) # With this new column, every proposal will have a unique identifier for the DAO it belongs to, regardless of whether the 'deployment_id' or 'platform_deployment_id' is available.

proposals["dao_id"].isna().sum()

np.int64(0)

In [ ]:
# Now, let's identify proposals that have a 'deployment_id' but that 'deployment_id' does not exist in the deployments dataset. 
# These are "orphan" proposals, as they reference a deployment that is not present in our deployments dataset.
orphan_proposals = proposals[
    proposals["deployment_id"].notna() 
    & ~proposals["deployment_id"].isin(deployments["deployment_id"])
]

In [74]:
orphan_proposals.shape # Show the number of orphan proposals

(37954, 9)

In [77]:
orphan_proposals["platform"].value_counts() # Show the count of orphan proposals by platform

platform
tally       37635
snapshot      319
Name: count, dtype: int64

In [ ]:
proposals[proposals["author"].isna()]["platform"].value_counts() # Which platforms have missing 'author' values in the proposals dataset?

platform
tally     37839
realms    21153
Name: count, dtype: int64

In [78]:
proposals["author"].nunique()

145348

In [ ]:
# Identify duplicated proposals
duplicated_proposals = proposals[
    proposals["proposal_id"].duplicated(keep=False)
]

In [44]:
duplicated_proposals.shape

(30996, 9)

In [45]:
duplicated_proposals["platform"].value_counts()

platform
tally    30996
Name: count, dtype: int64

In [46]:
proposals.duplicated().sum()

np.int64(5823)

In [ ]:
# Describe the statistics for the deployment metrics
deployments[["proposals_count", "unique_voters", "votes_count"]].describe()

,proposals_count,unique_voters,votes_count
count,86489.000000,86489.000000,8.648900e+04
mean,3.913908,31.911179,7.780829e+02
std,105.007523,994.182513,1.015083e+05
min,0.000000,0.000000,0.000000e+00
25%,1.000000,1.000000,1.000000e+00
50%,1.000000,1.000000,1.000000e+00
75%,2.000000,2.000000,2.000000e+00
max,29681.000000,138480.000000,2.231116e+07


In [86]:
# Count deployments with no unique voters
(deployments["unique_voters"] == 0).sum().item() # type: ignore

19826

In [ ]:
# Count deployments with at least 10 unique voters
(deployments["unique_voters"] >= 10).sum().item() # type: ignore

7886

In [ ]:
# Count deployments with at least 100 unique voters
(deployments["unique_voters"] >= 100).sum().item() # type: ignore

2271

In [51]:
dates = pd.to_datetime(proposals["date"], errors="coerce")

In [85]:
dates.min(), dates.max()

(Timestamp('2018-10-27 13:47:38'), Timestamp('2025-11-21 12:07:23'))

In [ ]:
# Count the number of missing dates in the proposals dataset
dates.isna().sum().item() # type: ignore

79

In [ ]:
linked = proposals.dropna(subset=["dao_id", "author"]) # drop proposals with missing 'dao_id' or 'author'

In [ ]:
daos_per_author = linked.groupby("author")["dao_id"].nunique()  # count the number of unique DAOs each author has submitted proposals to

In [ ]:
# Number of authors who have submitted proposals to at least 2 unique DAOs
# These are authors who have submitted proposals to multiple DAOs, indicating a broader engagement in the ecosystem.
(daos_per_author >= 2).sum().item() # type: ignore 

8593

In [ ]:
multi_dao_authors = daos_per_author[daos_per_author >= 2].index # Get the authors who have submitted proposals to at least 2 unique DAOs

In [ ]:
multi = linked[linked["author"].isin(multi_dao_authors)] # Filter the proposals dataset to include only proposals from authors who have submitted to multiple DAOs

In [ ]:
# Count the number of unique edges (interactions between DAOs)
edges = Counter()

for daos in multi.groupby("author")["dao_id"].unique():
    for a, b in itertools.combinations(sorted(daos), 2):
        edges[(a, b)] += 1

len(edges)

28253

In [60]:
nodes = set(itertools.chain.from_iterable(edges))

In [61]:
len(nodes)

7971